In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import pytz

In [2]:
local_tz = pytz.timezone('Asia/Jakarta')  # Replace with your time zone
now_local = datetime.now(local_tz)
date_format = "%m-%d-%Y" # Choose your desired format
date_str = now_local.strftime(date_format)

In [3]:
from google.colab import files
file = files.upload()

Saving _report__driver_assigned_order__historical__2026-03-23T11_26_50.897112511Z.csv to _report__driver_assigned_order__historical__2026-03-23T11_26_50.897112511Z.csv


In [4]:
# Get the filename from the uploaded file dictionary
uploaded_filename = list(file.keys())[0]

df = pd.read_csv(uploaded_filename)
df_backup = df.copy(deep=True)

df_columns = ['delivery_date',  'hubs', 'time_slot', 'total_weight_perorder', 'order_no']

df_filtered = df[df_columns].copy() # Create a copy to avoid SettingWithCopyWarning

# Convert 'delivery_date' to datetime objects
df_filtered['delivery_date'] = pd.to_datetime(df_filtered['delivery_date'], errors='coerce')


# Add 'week' and 'month' columns
df_filtered.loc[:, 'week'] = df_filtered['delivery_date'].dt.isocalendar().week
df_filtered.loc[:, 'month'] = df_filtered['delivery_date'].dt.month

df_filtered.loc[:, 'time_slot'] = df_filtered['time_slot'].str.replace('slot-12bb', 'slot-1')
df_filtered.loc[:, 'time_slot'] = df_filtered['time_slot'].str.replace('slot-1bb', 'slot-1')
df_filtered.loc[:, 'time_slot'] = df_filtered['time_slot'].str.replace('slot-0bb', 'slot-0')
df_filtered.loc[:, 'time_slot'] = df_filtered['time_slot'].str.replace('slot-b2b-2', 'slot-0')
df_filtered.loc[:, 'time_slot'] = df_filtered['time_slot'].str.replace('slot-13', 'slot-sameday')
df_filtered.loc[:, 'time_slot'] = df_filtered['time_slot'].str.replace('slot-sameday03', 'slot-2')
# Sort by multiple columns:
sorted_df = df_filtered.sort_values(by=['hubs', 'delivery_date', 'time_slot'], ascending=[True, True,True])

sorted_df.head()

,delivery_date,hubs,time_slot,total_weight_perorder,order_no,week,month
3016,2026-03-16,Beji - Depok,slot-0,8.0,DH-ZZQ5R40A1HXB-NR,12,3
3018,2026-03-16,Beji - Depok,slot-0,1.0,DH-ZZB4Y3MM0DUT-NR,12,3
3179,2026-03-16,Beji - Depok,slot-0,7.0,DH-XCP4N4GMQ9EC-NR,12,3
3181,2026-03-16,Beji - Depok,slot-0,8.0,DH-XBREX4NY4VUC-NR,12,3
3252,2026-03-16,Beji - Depok,slot-0,2.0,DH-W5CLK4XVKL3V-NR,12,3


In [5]:
# Create the pivot table
pivot_table = sorted_df.pivot_table(values=['total_weight_perorder', 'order_no'], index=['delivery_date', 'hubs', 'week', 'month'], columns='time_slot', aggfunc={'total_weight_perorder': 'sum', 'order_no': 'count'})

# Reset the index to make 'week' and 'month' regular columns
pivot_table_reset = pivot_table.reset_index()

# Flatten the MultiIndex columns more carefully
new_column_names = []
for col in pivot_table_reset.columns:
    if isinstance(col, tuple):
        # Join non-empty parts of the tuple with an underscore
        # This handles cases like ('delivery_date', '') becoming 'delivery_date'
        cleaned_name = '_'.join(filter(None, col))
        new_column_names.append(cleaned_name)
    else:
        new_column_names.append(col) # Keep single-level columns as is

pivot_table_reset.columns = new_column_names

# Get the list of columns
cols = pivot_table_reset.columns.tolist()

# Identify the columns to move (these should now be 'week' and 'month' without underscores)
cols_to_move = ['week', 'month']
cols_to_keep = [col for col in cols if col not in cols_to_move]

# Create the new ordered list of columns
new_order = cols_to_keep + cols_to_move

# Reindex the DataFrame with the new column order
pivot_table_ordered = pivot_table_reset[new_order]

# Set the index back to delivery_date and hubs (using original names)
pivot_table = pivot_table_ordered.set_index(['delivery_date', 'hubs'])

# Print the pivot table
pivot_table

order_no_slot-0  order_no_slot-1  \
delivery_date hubs                                                       
2026-03-16    Beji - Depok                       47.0             33.0   
              Bekasi - CIBUBUR                   34.0             28.0   
              Bintaro - TANGSEL                  46.0             30.0   
              Cibitung - KAB BEKASI              53.0             26.0   
              Karawaci - TANGERANG               42.0             21.0   
              Kembangan - JAKBAR                 59.0             53.0   
              Kramat - JAKPUS                    84.0             87.0   
              Pejaten - JAKSEL                   72.0             70.0   
              Pondok Kopi - JAKTIM               57.0             36.0   
              Satellite HUB Bogor                27.0             26.0   
              Satellite HUB Karawaci             20.0              5.0   
              Sentul - BOGOR                     28.0             20.0   
2026-03-17    Beji - Depok                       48.0             32.0   
              Bekasi - CIBUBUR                   44.0             20.0   
              Bintaro - TANGSEL                  41.0             26.0   
              Cibitung - KAB BEKASI              49.0             21.0   
              Karawaci - TANGERANG               52.0             29.0   
              Kembangan - JAKBAR                 60.0             60.0   
              Kramat - JAKPUS                    67.0             61.0   
              Pejaten - JAKSEL                   66.0             71.0   
              Pondok Kopi - JAKTIM               43.0             37.0   
              Satellite HUB Bogor                38.0             30.0   
              Satellite HUB Karawaci             20.0              8.0   
              Sentul - BOGOR                     29.0             24.0   
2026-03-18    Beji - Depok                       45.0             41.0   
              Bekasi - CIBUBUR                   41.0             19.0   
              Bintaro - TANGSEL                  35.0             19.0   
              Cibitung - KAB BEKASI              56.0             42.0   
              Karawaci - TANGERANG               38.0             37.0   
              Kembangan - JAKBAR                 48.0             62.0   
              Kramat - JAKPUS                    51.0             69.0   
              Pejaten - JAKSEL                   66.0             65.0   
              Pondok Kopi - JAKTIM               49.0             49.0   
              Satellite HUB Bogor                33.0             31.0   
              Satellite HUB Karawaci             20.0             12.0   
              Sentul - BOGOR                     23.0             39.0   
2026-03-19    Beji - Depok                       55.0             44.0   
              Bekasi - CIBUBUR                   57.0             21.0   
              Bintaro - TANGSEL                  55.0             27.0   
              Cibitung - KAB BEKASI              52.0             34.0   
              Karawaci - TANGERANG               50.0             34.0   
              Kembangan - JAKBAR                 53.0             64.0   
              Kramat - JAKPUS                    72.0             74.0   
              Pejaten - JAKSEL                   80.0             75.0   
              Pondok Kopi - JAKTIM               61.0             33.0   
              Satellite HUB Bogor                61.0             36.0   
              Satellite HUB Karawaci             35.0             10.0   
              Sentul - BOGOR                     46.0             29.0   

                                      order_no_slot-2  order_no_slot-sameday  \
delivery_date hubs                                                             
2026-03-16    Beji - Depok                       18.0                    NaN   
              Bekasi - CIBUBUR                   22.0                    NaN   
              Bin

In [6]:
pivot_table.shape

(48, 10)

In [7]:
pivot_table.to_csv('fleet_capacity_utilization.csv', index=True, header=False)
files.download('fleet_capacity_utilization.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>